# Tool-call guardrail: train + evaluate on Kaggle (T4)

Settings → Accelerator: **GPU T4 x1**, Internet: **on**. Add the `guardrail-bundle` dataset (made by `kaggle/make_bundle.sh`) via *Add Input*. Optional: add a Kaggle secret `HF_TOKEN` to push artifacts to Hugging Face.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # single GPU: avoids DataParallel quirks with LoRA + checkpointing
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q 'transformers==5.18.0' 'peft==0.21.2' accelerate datasets datasketch   # versions validated locally
import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.is_available())

In [ ]:
# locate the uploaded bundle and copy it to a writable dir
import glob, shutil
src = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/scripts/train.py', recursive=True)[0]))
shutil.copytree(src, '/kaggle/working/repo', dirs_exist_ok=True)
%cd /kaggle/working/repo
!ls data/processed && cat data/processed/stats.json | head -30

## 1. Zero-shot baselines (before any training)
Prompted LLM judges from the same family + an off-the-shelf prompt-injection classifier.

In [ ]:
!python scripts/baselines_zero_shot.py --judge Qwen/Qwen3-0.6B Qwen/Qwen3-1.7B --protectai --splits test

## 2. Main detector: Qwen3-0.6B-Base + LoRA classification head

In [ ]:
!python scripts/train.py --model Qwen/Qwen3-0.6B-Base --out runs/qwen3-0.6b-lora --max-len 1024 --bs 8 --grad-accum 2 --lr 2e-4 --epochs 3

In [ ]:
!python scripts/evaluate.py --model runs/qwen3-0.6b-lora

## 3. Alternative: ModernBERT-base encoder, full fine-tune (the comparison the brief asks us to justify against)

In [ ]:
!python scripts/train.py --model answerdotai/ModernBERT-base --out runs/modernbert-base --max-len 1024 --bs 16 --grad-accum 1 --lr 3e-5 --epochs 4

In [ ]:
!python scripts/evaluate.py --model runs/modernbert-base

## 4. Save artifacts
Download `outputs.zip` from the Output tab, or push to the Hub if `HF_TOKEN` is set.

In [ ]:
!cd /kaggle/working && zip -qr outputs.zip repo/results repo/runs/*/adapter_* repo/runs/*/*.json repo/runs/*/*.safetensors repo/runs/*/tokenizer* -x '*/checkpoint-*' && ls -lh outputs.zip
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded: run scripts/push_hf.py to upload')
except Exception as e:
    print('no HF_TOKEN secret:', e)